In [1]:
# !pip install -r requirements.txt
import importlib
importlib.reload
# Load the TensorBoard notebook extension
%load_ext tensorboard
import torch, os, sys
print("torch version  :", torch.__version__)

torch version  : 2.9.0.dev20250715


In [1]:
!python scripts/convert_dm4.py --input data/Diffraction_SI.dm4 --output data/train_tensor.pt --downsample 8 --mode bin 

Saved 4788 patterns of size 64*64 → data/train_tensor.pt


In [3]:
!python -m scripts.train --data data/train_tensor.pt --epochs 1 --batch 128 --latent 8 --lr 0.00003 --output_dir outputs --device mps --summary True

Using device: mps
Seed set to 42
Detected input size: 64x64
/Users/louisg/.pyenv/versions/custom_ae/lib/python3.11/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
────────────────────────────────────────────────────────────────────────────────
Layer                              Input → Output                    Params
────────────────────────────────────────────────────────────────────────────────
000_model_encoder_conv_input_Conv2d(1, 1, 64, 64) → (1, 64, 64, 64)       640
001_model_encoder_bn_input_BatchNorm2d(1, 64, 64, 64) → (1, 64, 64, 64)       128
002_model_encoder_conv_pre_Conv2d  (1, 64, 64, 64) → (1, 128, 64, 64)    73,856
003_model_encoder_bn_pre_BatchNorm2d(1, 128, 64, 64) → (1, 128, 64, 64)       256
004_model_encoder_resnet1_conv_block_conv1_Conv2d(1, 128, 64, 64) → (1, 128, 64, 64)   147,584
005_model_encoder_resnet1_conv_bl

In [4]:
!python -m scripts.generate_embeddings --input data/train_tensor.pt --checkpoint outputs/ae.ckpt --batch_size 2048 --output outputs/embeddings.pt

────────────────────────────────────────────────────────────────────────────────
Layer                              Input → Output                    Params
────────────────────────────────────────────────────────────────────────────────
000_conv_input_Conv2d              (1, 1, 64, 64) → (1, 64, 64, 64)       640
001_bn_input_BatchNorm2d           (1, 64, 64, 64) → (1, 64, 64, 64)       128
002_conv_pre_Conv2d                (1, 64, 64, 64) → (1, 128, 64, 64)    73,856
003_bn_pre_BatchNorm2d             (1, 128, 64, 64) → (1, 128, 64, 64)       256
004_resnet1_conv_block_conv1_Conv2d(1, 128, 64, 64) → (1, 128, 64, 64)   147,584
005_resnet1_conv_block_bn1_BatchNorm2d(1, 128, 64, 64) → (1, 128, 64, 64)       256
006_resnet1_conv_block_conv2_Conv2d(1, 128, 64, 64) → (1, 128, 64, 64)   147,584
007_resnet1_conv_block_bn2_BatchNorm2d(1, 128, 64, 64) → (1, 128, 64, 64)       256
008_resnet1_conv_block_conv3_Conv2d(1, 128, 64, 64) → (1, 128, 64, 64)   147,584
009_resnet1_conv_block_bn3_BatchN

In [16]:
import torch, numpy as np
raw = torch.load("data/train_tensor.pt")
print("raw tensor shape:", raw.shape)         # (N, 1, Qy, Qx)
N = raw.shape[0]
print("number of probe positions:", N)

# quick factor search
cands = [(f, N//f) for f in range(1, int(np.sqrt(N))+1) if N % f == 0]
print("factor pairs:", cands[:10], "…")


raw tensor shape: torch.Size([4788, 1, 64, 64])
number of probe positions: 4788
factor pairs: [(1, 4788), (2, 2394), (3, 1596), (4, 1197), (6, 798), (7, 684), (9, 532), (12, 399), (14, 342), (18, 266)] …


In [5]:
# scan, bright‑field background, every latent dim in one mosaic
!python scripts/visualise_scan_latents.py \
       --raw data/train_tensor.pt \
       --latents outputs/embeddings.pt \
       --scan 42 114 \
       --virtual bf \
       --lat_max_cols 6 \
       --outfig outputs/latent_mosaic.png



/Users/louisg/PycharmProjects/m3-learning-m3_learning-dc4a2d5/LTSJ_exp/Custom_4DSTEM_AE/scripts/visualise_scan_latents.py:154: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
Saved outputs/latent_mosaic.png
